# ResNet-50 at 320x320

## 1. Setup

In [ ]:
import gc
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError("Folder src not found. Let's open notebook from this repo.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

## 2. Import shared code

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset

from src.datasets.glaucoma_dataset import GlaucomaDataset, IDX_TO_CLASS, _drop_duplicates
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (
    CHECKPOINT_DIR, DEVICE, LEARNING_RATE, NUM_CLASSES, NUM_EPOCHS,
    SEED, WEIGHT_DECAY, set_seed
)
from src.training.train_loop import evaluate
from src.utils.metrics import compute_metrics, save_metrics, print_report
from src.utils.visualize import plot_training_curves, plot_confusion_matrix

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in this notebook kernel')

set_seed()

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
IMAGE_SIZE = 320
BATCH_SIZE_CANDIDATES = (16, 8, 4)  # falls back to a smaller batch size on CUDA OOM
AMP_ENABLED = True
EPOCHS = NUM_EPOCHS
PATIENCE = 5
MODEL_NAME = 'resnet50'
CLASS_NAMES = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / f'{MODEL_NAME}_best.pt'

## 4. 320x320 data pipeline

In [ ]:
train_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

raw_dataset = load_dataset('moondream/glaucoma-detection')
train_set = _drop_duplicates(raw_dataset['train'])
val_set = _drop_duplicates(raw_dataset['validation'])
test_set = _drop_duplicates(raw_dataset['test'])

def build_experiment_loaders(batch_size):
    train_data = GlaucomaDataset(train_set, transform=train_transform_320)
    val_data = GlaucomaDataset(val_set, transform=eval_transform_320)
    test_data = GlaucomaDataset(test_set, transform=eval_transform_320)
    return (
        DataLoader(train_data, batch_size=batch_size, shuffle=True),
        DataLoader(val_data, batch_size=batch_size, shuffle=False),
        DataLoader(test_data, batch_size=batch_size, shuffle=False),
    )

## 5. Train with AMP and an automatic OOM fallback

In [ ]:
def fit_resnet50(model, optimizer, loss_fn, train_loader, val_loader):
    history = {'train_loss': [], 'val_loss': []}
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_val_loss = float('inf')
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_train_loss += loss.item() * images.size(0)
        train_loss = total_train_loss / len(train_loader.dataset)

        val_loss, val_preds, val_labels = evaluate(model, val_loader, loss_fn)
        val_metrics = compute_metrics(val_labels, val_preds)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        improved = val_loss < best_val_loss
        print(
            f"Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f} | "
            f"val_loss: {val_loss:.4f} - val_acc: {val_metrics['accuracy']:.4f} | "
            f"val_f1_macro: {val_metrics['f1_macro']:.4f}"
            + ('  (best validation loss)' if improved else '')
        )

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f'No validation-loss improvement for {PATIENCE} epochs; stopping early.')
                break

    return history, best_epoch

In [ ]:
oom_occurred = False
for batch_size in BATCH_SIZE_CANDIDATES:
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(SEED)
    train_loader, val_loader, test_loader = build_experiment_loaders(batch_size)

    set_seed(SEED)  # Sanity-check iteration must not change training initialization/order.
    model = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, freeze_features=False).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    print(f"\nAttempting training with batch size {batch_size}...")
    try:
        history, best_epoch = fit_resnet50(model, optimizer, loss_fn, train_loader, val_loader)
        actual_batch_size = batch_size
        break
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower():
            raise
        oom_occurred = True
        print(f'CUDA OOM at batch size {batch_size}; retrying with a smaller batch size.')
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    raise RuntimeError(f'CUDA OOM at all batch sizes {BATCH_SIZE_CANDIDATES}.')

print(f'\nBest epoch: {best_epoch} | Batch size used: {actual_batch_size} | OOM occurred: {oom_occurred}')
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'

## 6. Training curves

In [ ]:
fig = plot_training_curves(history, model_name=MODEL_NAME)
fig.show()

## 7. Reload the BEST checkpoint and evaluate on the test set

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))

test_loss, test_preds, test_labels = evaluate(model, test_loader, loss_fn)
metrics = compute_metrics(test_labels, test_preds)

print(f"Test loss : {test_loss:.4f}")
print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"F1 macro  : {metrics['f1_macro']:.4f}")
print()
print_report(test_labels, test_preds)

save_metrics(metrics, model_name=MODEL_NAME)

## 8. Confusion matrix

In [ ]:
fig_cm = plot_confusion_matrix(metrics['confusion_matrix'], model_name=MODEL_NAME)
fig_cm.show()